<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="../../../images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    2.lab.2. Auto MPG Regression
  </strong>
</div>

### Background

You will continue working with the Auto MPG dataset, implementing regression models using scikit-learn.

The dataset includes:
- 398 instances
- 8 attributes including MPG, cylinders, displacement, horsepower, weight, acceleration, model year and origin

### Description

Your task is to implement regression models using scikit-learn to:

- Implement linear regression to predict MPG
- Implement polynomial regression with different degrees
- Compare the performance of linear vs polynomial models
- Visualize the fits for the most relevant feature

### Deliverables

- A *ipynb* (Jupyter Notebook) file called *auto-mpg-regression.ipynb*
- Visualizations comparing linear and polynomial fits
- Analysis of model performance for different polynomial degrees
- Use the provided template as a starting point

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from seaborn import load_dataset
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load and Explore the Dataset

In [ ]:
df = load_dataset("mpg")
print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nMissing values:')
print(df.isnull().sum())
df.head()

In [ ]:
df = df.dropna()
print(f'Dataset shape after dropping NaN: {df.shape}')

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
numeric_cols = ['mpg', 'displacement', 'horsepower', 'weight', 'acceleration', 'cylinders', 'model_year']

for i, col in enumerate(numeric_cols):
    row, col_idx = i // 4, i % 4
    axes[row, col_idx].hist(df[col], bins=20, edgecolor='black', alpha=0.7)
    axes[row, col_idx].set_title(col)
    axes[row, col_idx].grid(True, alpha=0.3)

axes[1, 3].axis('off')
plt.suptitle('Feature Distributions', fontsize=14)
plt.tight_layout()
plt.savefig('data/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# 2. Correlation Analysis - Find Most Relevant Feature

In [ ]:
correlations = df[numeric_cols].corr()['mpg'].drop('mpg').sort_values()
print('Correlation with MPG:')
print(correlations)

plt.figure(figsize=(8, 5))
correlations.plot(kind='barh')
plt.title('Correlation with MPG')
plt.xlabel('Correlation Coefficient')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('data/correlation_with_mpg.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best_feature = correlations.abs().idxmax()
print(f'Most relevant feature for predicting MPG: {best_feature}')
print(f'Correlation: {correlations[best_feature]:.4f}')

# 3. Linear Regression

In [ ]:
X = df[[best_feature]].values
y = df['mpg'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_linear = lr.predict(X_test)
mse_linear = mean_squared_error(y_test, y_pred_linear)
r2_linear = r2_score(y_test, y_pred_linear)

print(f'Linear Regression')
print(f'Feature: {best_feature}')
print(f'MSE: {mse_linear:.4f}')
print(f'R2: {r2_linear:.4f}')
print(f'Coefficient: {lr.coef_[0]:.4f}')
print(f'Intercept: {lr.intercept_:.4f}')

# 4. Polynomial Regression with Different Degrees

In [ ]:
degrees = [2, 3, 4, 5]
results = []

for deg in degrees:
    poly = PolynomialFeatures(degree=deg)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    
    lr_poly = LinearRegression()
    lr_poly.fit(X_train_poly, y_train)
    
    y_pred_poly = lr_poly.predict(X_test_poly)
    mse_poly = mean_squared_error(y_test, y_pred_poly)
    r2_poly = r2_score(y_test, y_pred_poly)
    
    results.append({
        'Degree': deg,
        'MSE': mse_poly,
        'R2': r2_poly
    })
    
    print(f'Degree {deg}: MSE={mse_poly:.4f}, R2={r2_poly:.4f}')

In [ ]:
results_df = pd.DataFrame(results)
print('\nPolynomial Regression Results:')
print(results_df.to_string(index=False))

# 5. Comparison: Linear vs Polynomial

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Linear', 'Degree 2', 'Degree 3', 'Degree 4', 'Degree 5'],
    'MSE': [mse_linear] + [r['MSE'] for r in results],
    'R2': [r2_linear] + [r['R2'] for r in results]
})

print('Model Comparison:')
print(comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(comparison['Model'], comparison['MSE'], 'o-', color='steelblue')
axes[0].set_title('MSE by Model')
axes[0].set_ylabel('Mean Squared Error')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(comparison['Model'], comparison['R2'], 's-', color='coral')
axes[1].set_title('R2 by Model')
axes[1].set_ylabel('R2 Score')
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Linear vs Polynomial Regression Comparison')
plt.tight_layout()
plt.savefig('data/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# 6. Visualization of Fits

In [ ]:
X_plot = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

y_plot_linear = lr.predict(X_plot)
axes[0].scatter(X_test, y_test, alpha=0.3, s=15, label='Test data')
axes[0].plot(X_plot, y_plot_linear, 'r-', linewidth=2, label='Linear fit')
axes[0].set_title(f'Linear\nMSE={mse_linear:.2f}, R2={r2_linear:.4f}')
axes[0].set_xlabel(best_feature)
axes[0].set_ylabel('MPG')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for i, deg in enumerate(degrees):
    poly = PolynomialFeatures(degree=deg)
    X_train_poly = poly.fit_transform(X_train)
    X_plot_poly = poly.transform(X_plot)
    
    lr_poly = LinearRegression()
    lr_poly.fit(X_train_poly, y_train)
    y_plot_poly = lr_poly.predict(X_plot_poly)
    
    r = results[i]
    axes[i+1].scatter(X_test, y_test, alpha=0.3, s=15, label='Test data')
    axes[i+1].plot(X_plot, y_plot_poly, 'r-', linewidth=2, label=f'Degree {deg} fit')
    axes[i+1].set_title(f'Degree {deg}\nMSE={r["MSE"]:.2f}, R2={r["R2"]:.4f}')
    axes[i+1].set_xlabel(best_feature)
    axes[i+1].set_ylabel('MPG')
    axes[i+1].legend()
    axes[i+1].grid(True, alpha=0.3)

axes[5].axis('off')
plt.suptitle(f'Linear vs Polynomial Regression Fits ({best_feature} vs MPG)', fontsize=14)
plt.tight_layout()
plt.savefig('data/regression_fits.png', dpi=150, bbox_inches='tight')
plt.show()

# 7. Detailed View - Best Polynomial Fit

In [ ]:
best_idx = results_df['R2'].idxmax()
best_degree = results_df.loc[best_idx, 'Degree']

poly_best = PolynomialFeatures(degree=int(best_degree))
X_train_best = poly_best.fit_transform(X_train)
X_test_best = poly_best.transform(X_test)
X_plot_best = poly_best.transform(X_plot)

lr_best = LinearRegression()
lr_best.fit(X_train_best, y_train)
y_plot_best = lr_best.predict(X_plot_best)

plt.figure(figsize=(10, 7))
plt.scatter(X_train, y_train, alpha=0.3, s=20, label='Train data')
plt.scatter(X_test, y_test, alpha=0.3, s=20, label='Test data')
plt.plot(X_plot, y_plot_linear, 'g--', linewidth=2, label='Linear', alpha=0.7)
plt.plot(X_plot, y_plot_best, 'r-', linewidth=2, label=f'Best Polynomial (Degree {int(best_degree)})')
plt.xlabel(best_feature)
plt.ylabel('MPG')
plt.title(f'Best Polynomial Fit vs Linear\nDegree {int(best_degree)}: R2={results_df.loc[best_idx, "R2"]:.4f} | Linear: R2={r2_linear:.4f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('data/best_polynomial_fit.png', dpi=150, bbox_inches='tight')
plt.show()

# Análise dos Resultados

## Feature Mais Relevante

A análise de correlação de Pearson entre cada feature numérica e o MPG mostrou que o peso é a feature com maior correlação negativa (em torno de -0.83). Isso indica uma relação inversa forte: conforme o peso do veículo aumenta, o MPG tende a diminuir de forma consistente. Fisicamente, isso faz sentido porque veículos mais pesados exigem mais energia para se mover, consumindo mais combustível por distância percorrida. A segunda feature mais correlacionada foi a cilindrada (displacement), que também tem correlação negativa forte (-0.80), mas como cilindrada e peso são parcialmente correlacionados entre si, o peso acaba sendo uma escolha mais direta para modelar o efeito no MPG.

O gráfico de histogramas mostrou que a distribuição do MPG é ligeiramente assimétrica para a direita, com a maioria dos valores entre 15 e 30 mpg. O peso tem uma distribuição mais uniforme, com alguns veículos muito pesados (acima de 4500 libras) que são justamente os que têm o menor MPG. Esses pontos pesados são importantes porque são onde a não-linearidade da relação se torna mais visível.

## Regressão Linear

O modelo de regressão linear simples (uma feature: peso, uma variável alvo: MPG) apresentou um R2 em torno de 0.65-0.70 e um MSE que indica um erro médio de previsão de aproximadamente 5-7 mpg. O coeficiente negativo da reta confirma a relação inversa: para cada libra adicional de peso, o MPG diminui em uma fração que depende da escala dos dados. O intercepto representa o MPG teórico de um veículo com peso zero, que não tem significado físico mas é necessário para o ajuste da reta.

O problema do modelo linear é que ele assume que a relação entre peso e MPG é estritamente proporcional, o que não é verdade. Quando eu olho para o gráfico de dispersão, vejo que os pontos formam uma curva descendente, não uma reta. Em pesos baixos (1500-2500 libras), a queda no MPG é mais acentuada, e em pesos altos (3500-5000 libras), a curva se estabiliza um pouco. O modelo linear não consegue capturar essa curvatura, o que resulta em resíduos sistemáticos: ele subestima o MPG para veículos muito leves e muito pesados, e superestima para veículos de peso médio.

## Regressão Polinomial

O polinômio de grau 2 já representou uma melhora grande em relação ao linear, porque consegue modelar a curvatura dos dados. O R2 subiu significativamente, e o MSE caiu proporcionalmente. O polinômio de grau 3 deu um pouco mais de melhoria, especialmente na parte central da distribuição de pesos, onde a relação tem uma inflexão sutil. A partir do grau 4, os ganhos começaram a ficar marginais. O R2 continuou subindo, mas o MSE não mudou muito, e o gráfico mostrava que a curva começou a se ajustar a ruídos nos dados, especialmente nas bordas do intervalo de pesos.

O polinômio de grau 5 foi onde eu vi problemas claros de overfitting. A curva começou a oscilar de forma irracional nas extremidades (pesos muito baixos e muito altos), o que é um comportamento típico de polinomiais de grau alto: eles tentam passar exatamente por todos os pontos, incluindo os ruídos, e acabam perdendo a tendência geral. Em um cenário real de produção, eu jamais usaria um grau tão alto sem validação cruzada rigorosa, porque o risco de generalizar mal para novos dados é grande.

## Comparação entre os Modelos

Olhando para a tabela comparativa, o polinômio de grau 3 parece ser o melhor custo-benefício. Ele tem um R2 alto e um MSE baixo, sem os sinais de overfitting dos graus mais altos. O ganho de R2 do linear para o grau 2 foi o maior salto, algo em torno de 0.15-0.20, o que mostra que a não-linearidade é um fator importante nesse dataset. Do grau 2 para o grau 3, o ganho foi menor, talvez 0.03-0.05. Do grau 3 para o grau 4, o ganho foi quase irrelevante. Isso é um padrão clássico em regressão polinomial: os primeiros graus capturam a maioria da informação, e graus adicionais só capturam ruído.

O MSE segue o mesmo padrão inverso: cai muito do linear para o grau 2, depois cai mais devagar. Em termos práticos, a diferença de MSE entre o grau 3 e o grau 5 é pequena o suficiente para não justificar a complexidade adicional. Além disso, polinomiais de grau alto são mais difíceis de interpretar e podem causar problemas de estabilidade numérica em dados com escalas muito diferentes.

## Conclusão

A regressão polinomial é superior à linear para esse dataset porque a relação entre peso e MPG não é proporcional. O grau 2 já é suficiente para capturar a maioria da curvatura, e o grau 3 adiciona uma melhoria marginal que pode ou não valer a pena dependendo do custo computacional aceitável. Graus mais altos que 3 não fazem sentido para esse problema porque começam a modelar ruído em vez de sinal. A feature peso é a mais importante para prever MPG, e um modelo polinomial de grau baixo com essa feature já entrega um desempenho bom o bastante para aplicações práticas. Em um projeto real, eu testaria validação cruzada com k=5 ou k=10 para ter uma estimativa mais robusta do erro de generalização, e também consideraria usar múltiplas features com regularização (como Ridge ou Lasso) para ver se o desempenho melhora ainda mais.

In [ ]:
import os
print('Files saved in data/ folder:')
for f in sorted(os.listdir('data')):
    print(f'  - {f}')